# Object State Classification: Robotic Kitchen Perception

**Pipeline position:** GroundingDINO → SAM 2 → **[this notebook]** + DA2 (depth) → SAM 2 temporal → spatial relation graph

This notebook runs the full experimental pipeline:
1. Zero-shot evaluation (SigLIP 2 & CLIP on ChangeIt-Frames)
2. Linear probe on frozen sim embeddings (full/empty)
3. Cross-domain transfer (sim → ChangeIt-Frames)
4. MLP adapter training across 6 state pairs
5. Visualisations and results summary

All heavy model inference is cached so re-running skips extraction automatically.

In [ ]:
import os, sys, pickle
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from tqdm.notebook import tqdm
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import average_precision_score, accuracy_score
from sklearn.model_selection import train_test_split
from transformers import AutoProcessor, AutoModel
import open_clip
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

ROOT = Path("..").resolve()
DATA = ROOT / "data"
RESULTS_CSV = ROOT / "results" / "csv"
RESULTS_PNG = ROOT / "results" / "png"
sys.path.insert(0, str(ROOT))

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Device: {device}")
print(f"ROOT: {ROOT}")

In [ ]:
BATCH_SIZE = 32

def extract_siglip_embeddings(images):
    """Extract SigLIP 2 So400m pooled embeddings from a list of PIL images."""
    proc  = AutoProcessor.from_pretrained("google/siglip2-so400m-patch16-384")
    model = AutoModel.from_pretrained("google/siglip2-so400m-patch16-384").to(device).eval()
    embs  = []
    for i in tqdm(range(0, len(images), BATCH_SIZE), desc="SigLIP 2"):
        batch  = images[i : i + BATCH_SIZE]
        inputs = proc(images=batch, return_tensors="pt", padding="max_length").to(device)
        with torch.no_grad():
            pooled = model.vision_model(pixel_values=inputs["pixel_values"]).pooler_output
        embs.append(pooled.cpu().float().numpy())
    del model
    if device == "mps": 
        torch.mps.empty_cache()
    return np.concatenate(embs)

def extract_clip_embeddings(images):
    """Extract CLIP ViT-L/14 normalised embeddings from a list of PIL images."""
    model, _, preprocess = open_clip.create_model_and_transforms("ViT-L-14", pretrained="openai")
    model = model.to(device).eval()
    embs  = []
    for i in tqdm(range(0, len(images), BATCH_SIZE), desc="CLIP"):
        batch   = images[i : i + BATCH_SIZE]
        tensors = torch.stack([preprocess(img) for img in batch]).to(device)
        with torch.no_grad():
            feats = model.encode_image(tensors)
            feats = feats / feats.norm(dim=-1, keepdim=True)
        embs.append(feats.cpu().float().numpy())
    del model
    if device == "mps": 
        torch.mps.empty_cache()
    return np.concatenate(embs)

## 1  Zero-shot Evaluation on ChangeIt-Frames

**Task:** binary state classification (initial state → terminal state) on 35 real-world object categories.

**Label scheme:** 0 = initial steady state, 3 = terminal steady state; transition frames (1, 2) skipped.

**Metric:** Average Precision (AP) per category, then mean AP, this way we're matching the Newman et al. (2024) protocol.

In [ ]:
from utils.classification_states import a  # {cat: {"initial": [...], "terminal": [...]}}

ANNOT_DIR = DATA / "annotations"
CROP_DIR  = DATA / "ChangeIT-Subset-Crop"

def build_changeit_dataset(liquid_only=False):
    """Returns {cat: [(img_path, binary_label), ...]} using only labels 0 and 3."""
    LIQUID_CATS = {"beer", "juice", "milk"}
    crop_folders = set(os.listdir(CROP_DIR))
    dataset = {}

    for cat in sorted(os.listdir(ANNOT_DIR)):
        cat_path = os.path.join(ANNOT_DIR, cat)
        if not os.path.isdir(cat_path) or cat not in a:
            continue
        if liquid_only and cat not in LIQUID_CATS:
            continue

        entries = []
        for csv_file in sorted(os.listdir(cat_path)):
            if not csv_file.endswith(".csv"):
                continue
            video_id = csv_file.split(".")[0]
            if video_id not in crop_folders:
                continue
            df = pd.read_csv(os.path.join(cat_path, csv_file), header=None, index_col=0)
            label_map = df[1].to_dict()
            crop_folder = os.path.join(CROP_DIR, video_id)
            for fname in sorted(os.listdir(crop_folder)):
                if not fname.endswith(".jpg"):
                    continue
                parts = fname.split("_")
                if len(parts) < 3:
                    continue
                try:
                    frame_idx = int(parts[2].split(".")[0])
                except ValueError:
                    continue
                label = label_map.get(frame_idx)
                if label == 0:
                    entries.append((os.path.join(crop_folder, fname), 0))
                elif label == 3:
                    entries.append((os.path.join(crop_folder, fname), 1))

        if entries and len({y for _, y in entries}) == 2:
            dataset[cat] = entries

    return dataset

dataset = build_changeit_dataset()
total_frames = sum(len(v) for v in dataset.values())
print(f"Categories: {len(dataset)}  |  Total frames: {total_frames}")

In [ ]:
def terminal_score_from_logits(logits, n_initial):
    init_score = logits[:, :n_initial].max(dim=1).values
    term_score = logits[:, n_initial:].max(dim=1).values
    return torch.softmax(torch.stack([init_score, term_score], dim=1), dim=1)[:, 1].cpu().numpy()

def eval_siglip_zeroshot(cat, entries, model, processor):
    init_prompts = a[cat]["initial"]
    term_prompts = a[cat]["terminal"]
    all_prompts = init_prompts + term_prompts
    n_initial = len(init_prompts)
    scores, labels = [], []
    for i in range(0, len(entries), BATCH_SIZE):
        batch = entries[i : i + BATCH_SIZE]
        paths, ys = zip(*batch)
        images = [Image.open(p).convert("RGB") for p in paths]
        inputs = processor(text=all_prompts, images=images,
                           return_tensors="pt", padding="max_length").to(device)
        with torch.no_grad():
            logits = model(**inputs).logits_per_image
        scores.extend(terminal_score_from_logits(logits, n_initial))
        labels.extend(ys)
    return average_precision_score(labels, scores)

def eval_clip_zeroshot(cat, entries, model, preprocess, tokenizer):
    init_prompts = a[cat]["initial"]
    term_prompts = a[cat]["terminal"]
    all_prompts = init_prompts + term_prompts
    n_initial = len(init_prompts)
    text_tokens = tokenizer(all_prompts).to(device)
    with torch.no_grad():
        text_features = model.encode_text(text_tokens)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
    scores, labels = [], []
    for i in range(0, len(entries), BATCH_SIZE):
        batch = entries[i : i + BATCH_SIZE]
        paths, ys = zip(*batch)
        imgs = torch.stack([preprocess(Image.open(p).convert("RGB")) for p in paths]).to(device)
        with torch.no_grad():
            img_features = model.encode_image(imgs)
            img_features = img_features / img_features.norm(dim=-1, keepdim=True)
            logits = (img_features @ text_features.T) * model.logit_scale.exp()
        scores.extend(terminal_score_from_logits(logits, n_initial))
        labels.extend(ys)
    return average_precision_score(labels, scores)

In [ ]:
ZS_CACHE = RESULTS_CSV / "results_zero_shot.csv"

if ZS_CACHE.exists():
    print("Loading cached zero-shot results...")
    zs_df = pd.read_csv(ZS_CACHE)
    siglip_aps = dict(zip(zs_df[zs_df.category != "MEAN"]["category"],
                          zs_df[zs_df.category != "MEAN"]["siglip2_ap"]))
    clip_aps = dict(zip(zs_df[zs_df.category != "MEAN"]["category"],
                        zs_df[zs_df.category != "MEAN"]["clip_ap"]))
else:
    print("Running zero-shot evaluation (slow — loads both models)...")
    siglip_processor = AutoProcessor.from_pretrained("google/siglip2-so400m-patch16-384")
    siglip_model = AutoModel.from_pretrained("google/siglip2-so400m-patch16-384").to(device).eval()
    siglip_aps = {}
    for cat, entries in tqdm(dataset.items(), desc="SigLIP 2 zero-shot"):
        siglip_aps[cat] = eval_siglip_zeroshot(cat, entries, siglip_model, siglip_processor)
    del siglip_model
    if device == "mps": 
        torch.mps.empty_cache()

    clip_model, _, clip_preprocess = open_clip.create_model_and_transforms("ViT-L-14", pretrained="openai")
    clip_tokenizer = open_clip.get_tokenizer("ViT-L-14")
    clip_model = clip_model.to(device).eval()
    clip_aps = {}
    for cat, entries in tqdm(dataset.items(), desc="CLIP zero-shot"):
        clip_aps[cat] = eval_clip_zeroshot(cat, entries, clip_model, clip_preprocess, clip_tokenizer)
    del clip_model

    all_cats = sorted(set(siglip_aps) | set(clip_aps))
    rows = [{"category": c, "n_frames": len(dataset[c]),
             "siglip2_ap": siglip_aps.get(c, float("nan")),
             "clip_ap": clip_aps.get(c, float("nan"))} for c in all_cats]
    rows.append({"category": "MEAN", "n_frames": total_frames,
                 "siglip2_ap": np.mean(list(siglip_aps.values())),
                 "clip_ap": np.mean(list(clip_aps.values()))})
    zs_df = pd.DataFrame(rows)
    zs_df.to_csv(ZS_CACHE, index=False)

mean_siglip_zs = zs_df[zs_df.category == "MEAN"]["siglip2_ap"].values[0]
mean_clip_zs = zs_df[zs_df.category == "MEAN"]["clip_ap"].values[0]
liquid_cats = ["beer", "juice", "milk"]
liq_siglip_zs = zs_df[zs_df.category.isin(liquid_cats)]["siglip2_ap"].mean()
liq_clip_zs = zs_df[zs_df.category.isin(liquid_cats)]["clip_ap"].mean()

display(zs_df.style.format({"siglip2_ap": "{:.3f}", "clip_ap": "{:.3f}"}).highlight_max(
    subset=["siglip2_ap", "clip_ap"], axis=1, color="#d4f1c0"))
print(f"\nMean AP  —  SigLIP 2: {mean_siglip_zs:.3f}   CLIP: {mean_clip_zs:.3f}")
print(f"Liquid   —  SigLIP 2: {liq_siglip_zs:.3f}   CLIP: {liq_clip_zs:.3f}")

## 2  Linear Probe on Sim Crops (full / empty)

Train a logistic regression on frozen embeddings from AI2-THOR crops.
Answers: *do frozen features already contain state signal, and does supervision help?*

**Data:** 300 crops (150 full / 150 empty) of Bowl, Cup, Pot across 30 FloorPlan scenes.
**Split:** 80% train / 20% holdout, stratified.

In [ ]:
SIM_CROPS_PATH = DATA / "sim_crops.pkl"
SIM_CACHE_PATH = DATA / "embeddings_cache.npz"

with open(SIM_CROPS_PATH, "rb") as f:
    sim_samples = pickle.load(f)

sim_images = [s["image"] for s in sim_samples]
sim_labels = np.array([s["label"] for s in sim_samples])
sim_obj_types = [s["object_type"] for s in sim_samples]
print(f"Loaded {len(sim_samples)} crops: (empty={sum(sim_labels==0)}, full={sum(sim_labels==1)})")

if SIM_CACHE_PATH.exists():
    print(f"Loading cached embeddings → {SIM_CACHE_PATH.name}")
    _cache = np.load(SIM_CACHE_PATH)
    siglip_emb = _cache["siglip"]
    clip_emb = _cache["clip"]
else:
    print("Extracting embeddings (cached after first run)...")
    siglip_emb = extract_siglip_embeddings(sim_images)
    clip_emb = extract_clip_embeddings(sim_images)
    np.savez(SIM_CACHE_PATH, siglip=siglip_emb, clip=clip_emb)
    print(f"Saved → {SIM_CACHE_PATH}")

print(f"SigLIP 2 embeddings: {siglip_emb.shape}   CLIP embeddings: {clip_emb.shape}")

In [ ]:
def run_probe(name, embeddings, labels):
    X_tr, X_te, y_tr, y_te = train_test_split(
        embeddings, labels, test_size=0.2, random_state=42, stratify=labels)
    clf = LogisticRegression(max_iter=1000, C=1.0).fit(X_tr, y_tr)
    scores = clf.predict_proba(X_te)[:, 1]
    ap = average_precision_score(y_te, scores)
    acc = accuracy_score(y_te, clf.predict(X_te))
    print(f"{name:<30}  AP={ap:.3f}   Acc={acc:.3f}   (train={len(y_tr)}, test={len(y_te)})")
    return ap, acc, clf, X_te, y_te, clf.predict(X_te)

siglip_ap, siglip_acc, siglip_clf, siglip_Xte, siglip_yte, siglip_preds = run_probe(
    "SigLIP 2 linear probe", siglip_emb, sim_labels)
clip_ap, clip_acc, clip_clf, clip_Xte, clip_yte, clip_preds = run_probe(
    "CLIP linear probe", clip_emb, sim_labels)

lp_df = pd.DataFrame([
    {"model": "SigLIP 2", "zero_shot_ap_liquid": liq_siglip_zs,
     "probe_ap_sim": siglip_ap, "probe_acc_sim": siglip_acc},
    {"model": "CLIP", "zero_shot_ap_liquid": liq_clip_zs,
     "probe_ap_sim": clip_ap, "probe_acc_sim": clip_acc},
])
lp_df.to_csv(RESULTS_CSV / "results_linear_probe.csv", index=False)
print()
display(lp_df.style.format("{:.3f}", subset=["zero_shot_ap_liquid","probe_ap_sim","probe_acc_sim"]))

## 3  Cross-domain Transfer: Sim → ChangeIt-Frames

Train the linear probe on **all** 300 AI2-THOR sim crops, then test on the ChangeIt-Frames liquid subset (beer, juice, milk which are the closest real-world analogue to `full`/`empty`).

This directly measures the sim-to-real gap before any domain adaptation.

In [ ]:
CIT_CACHE_PATH = DATA / "changeit_liquid_cache.npz"

liquid_dataset = build_changeit_dataset(liquid_only=True)
cit_images, cit_labels, cit_cats = [], [], []
for cat, entries in liquid_dataset.items():
    for path, lbl in entries:
        cit_images.append(Image.open(path).convert("RGB"))
        cit_labels.append(lbl)
        cit_cats.append(cat)
cit_labels = np.array(cit_labels)
print(f"ChangeIt liquid: {len(cit_images)} frames  "
      f"(empty={sum(cit_labels==0)}, full={sum(cit_labels==1)})")
print(f"Per category: { {c: cit_cats.count(c) for c in liquid_dataset} }")

if CIT_CACHE_PATH.exists():
    print(f"\nLoading cached ChangeIt embeddings > {CIT_CACHE_PATH.name}")
    _cit = np.load(CIT_CACHE_PATH)
    cit_siglip = _cit["siglip"]
    cit_clip   = _cit["clip"]
else:
    print("\nExtracting ChangeIt embeddings...")
    cit_siglip = extract_siglip_embeddings(cit_images)
    cit_clip   = extract_clip_embeddings(cit_images)
    np.savez(CIT_CACHE_PATH, siglip=cit_siglip, clip=cit_clip)
    print(f"Saved > {CIT_CACHE_PATH}")

In [ ]:
def cross_domain_eval(train_emb, train_labels, test_emb, test_labels):
    clf = LogisticRegression(max_iter=1000, C=1.0).fit(train_emb, train_labels)
    scores = clf.predict_proba(test_emb)[:, 1]
    return (average_precision_score(test_labels, scores),
            accuracy_score(test_labels, clf.predict(test_emb)))

s_xd_ap, s_xd_acc = cross_domain_eval(siglip_emb, sim_labels, cit_siglip, cit_labels)
c_xd_ap, c_xd_acc = cross_domain_eval(clip_emb,   sim_labels, cit_clip,   cit_labels)

xd_df = pd.DataFrame([
    {"model": "SigLIP 2", "zs_ap_liquid": liq_siglip_zs, "probe_ap_sim": siglip_ap,
     "xd_ap_changeit": s_xd_ap, "xd_acc_changeit": s_xd_acc},
    {"model": "CLIP",     "zs_ap_liquid": liq_clip_zs,   "probe_ap_sim": clip_ap,
     "xd_ap_changeit": c_xd_ap, "xd_acc_changeit": c_xd_acc},
])
xd_df.to_csv(RESULTS_CSV / "results_cross_domain.csv", index=False)

fmt_cols = ["zs_ap_liquid","probe_ap_sim","xd_ap_changeit","xd_acc_changeit"]
display(xd_df.style.format("{:.3f}", subset=fmt_cols))

for row in xd_df.itertuples():
    gap = row.xd_ap_changeit - row.probe_ap_sim
    print(f"{row.model:<10}  sim AP={row.probe_ap_sim:.3f}  >  real AP={row.xd_ap_changeit:.3f}  Δ={gap:+.3f}")

## 4  MLP Adapter on 6 State Pairs

A small MLP head trained on top of frozen SigLIP 2 So400m features across all 6 available state pairs from the AI2-THOR dataset.

```
SigLIP 2 So400m (frozen, 1152-dim pooled output)
  → Linear(1152→512) → GELU → Dropout(0.1)
  → Linear(512→256)  → GELU → Dropout(0.1)
  → Linear(256→6)    one logit per state pair
```

**Loss:** `BCEWithLogitsLoss` with per-sample applicability mask. Only the head for each sample's own state pair receives a gradient.

**Data:** `sim_crops_all.pkl` contains ~960 crops across 6 pairs (80 per class per pair).

In [ ]:
ALL_CROPS_PATH = DATA / "sim_crops_all.pkl"
ALL_CACHE_PATH = DATA / "embeddings_all_cache.npz"
ADAPTER_PATH = DATA / "adapter.pt"
PAIR_NAMES = ["full_empty", "open_closed", "on_off", "cooked_raw", "dirty_clean", "broken_intact"]
NUM_PAIRS = len(PAIR_NAMES)
EMBED_DIM = 1152

with open(ALL_CROPS_PATH, "rb") as f:
    all_samples = pickle.load(f)

all_images = [s["image"] for s in all_samples]
all_labels = np.array([s["label"] for s in all_samples])
all_pair_idxs = np.array([s["pair_idx"] for s in all_samples])

print(f"Loaded {len(all_samples)} crops across {NUM_PAIRS} state pairs:")
for i, name in enumerate(PAIR_NAMES):
    mask = all_pair_idxs == i
    print(f"  [{i}] {name:<16}  neg={int((all_labels[mask]==0).sum()):>3}  pos={int((all_labels[mask]==1).sum()):>3}")

if ALL_CACHE_PATH.exists():
    print(f"\nLoading cached embeddings → {ALL_CACHE_PATH.name}")
    all_embeddings = np.load(ALL_CACHE_PATH)["embeddings"]
else:
    print("\nExtracting SigLIP 2 embeddings for all crops...")
    all_embeddings = extract_siglip_embeddings(all_images)
    np.savez(ALL_CACHE_PATH, embeddings=all_embeddings)
    print(f"Saved → {ALL_CACHE_PATH}")

print(f"Embeddings shape: {all_embeddings.shape}")

In [ ]:
strat_key = all_pair_idxs * 2 + all_labels
idx = np.arange(len(all_samples))
idx_train, idx_test = train_test_split(idx, test_size=0.2, random_state=42, stratify=strat_key)

X_train = all_embeddings[idx_train]
y_train = all_labels[idx_train]
p_train = all_pair_idxs[idx_train]
X_test = all_embeddings[idx_test]
y_test = all_labels[idx_test]
p_test = all_pair_idxs[idx_test]
print(f"Train: {len(idx_train)}   Test: {len(idx_test)}")

class StateDataset(Dataset):
    def __init__(self, X, y, p):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()
        self.p = torch.from_numpy(p).long()
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i], self.p[i]

class StateAdapter(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(EMBED_DIM, 512), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(512, 256),       nn.GELU(), nn.Dropout(0.1),
            nn.Linear(256, NUM_PAIRS),
        )
    def forward(self, x): return self.net(x)

def masked_bce_loss(logits, labels, pair_idxs):
    B = logits.shape[0]
    target = torch.full((B, NUM_PAIRS), -1.0, device=logits.device)
    for i in range(B):
        target[i, pair_idxs[i]] = labels[i]
    mask = (target >= 0).float()
    loss = F.binary_cross_entropy_with_logits(logits, target.clamp(0, 1), reduction="none")
    return (loss * mask).sum() / mask.sum().clamp(min=1)

In [ ]:
EPOCHS = 10
adapter = StateAdapter().to(device)
n_params = sum(p.numel() for p in adapter.parameters())
print(f"Adapter parameters: {n_params:,}  (~{n_params/1e6:.2f}M)")

if ADAPTER_PATH.exists():
    print(f"Loading pre-trained adapter → {ADAPTER_PATH.name}  (delete to retrain)")
    adapter.load_state_dict(torch.load(ADAPTER_PATH, map_location=device))
else:
    train_loader = DataLoader(StateDataset(X_train, y_train, p_train),
                              batch_size=32, shuffle=True)
    optimizer = AdamW(adapter.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
    print(f"Training for {EPOCHS} epochs...")
    for epoch in range(1, EPOCHS + 1):
        adapter.train()
        total_loss = 0.0
        for x, y, p in train_loader:
            x, y, p = x.to(device), y.to(device), p.to(device)
            optimizer.zero_grad()
            loss = masked_bce_loss(adapter(x), y, p)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(x)
        scheduler.step()
        print(f"  Epoch {epoch:>2}/{EPOCHS}  loss={total_loss/len(idx_train):.4f}")
    torch.save(adapter.state_dict(), ADAPTER_PATH)
    print(f"Saved → {ADAPTER_PATH}")

In [ ]:
adapter.eval()
test_loader = DataLoader(StateDataset(X_test, y_test, p_test), batch_size=32, shuffle=False)
all_logits, all_ys, all_ps = [], [], []
with torch.no_grad():
    for x, y, p in test_loader:
        all_logits.append(adapter(x.to(device)).cpu())
        all_ys.append(y)
        all_ps.append(p)

all_logits = torch.cat(all_logits).numpy()
all_ys = torch.cat(all_ys).numpy()
all_ps = torch.cat(all_ps).numpy()

adapter_rows = []
for i, name in enumerate(PAIR_NAMES):
    mask = all_ps == i
    if mask.sum() < 2 or len(np.unique(all_ys[mask])) < 2:
        continue
    scores = torch.sigmoid(torch.from_numpy(all_logits[mask, i])).numpy()
    gt = all_ys[mask]
    ap = average_precision_score(gt, scores)
    acc = float(((scores >= 0.5).astype(int) == gt.astype(int)).mean())
    adapter_rows.append({"pair": name, "n_test": int(mask.sum()), "adapter_ap": ap, "adapter_acc": acc})

adapter_df = pd.DataFrame(adapter_rows)
adapter_df.to_csv(RESULTS_CSV / "results_adapter.csv", index=False)
display(adapter_df.style.format({"adapter_ap": "{:.3f}", "adapter_acc": "{:.3f}"}))
print(f"\nMean AP: {adapter_df.adapter_ap.mean():.3f}   Mean Acc: {adapter_df.adapter_acc.mean():.3f}")

## 5  Visualisations

### 5.1  Zero-shot vs Linear Probe (full/empty)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
x = np.array([0, 1])
width = 0.32
colors = {"SigLIP 2": "#4C72B0", "CLIP": "#DD8452"}

bars = [
    ax.bar(x[0] - width/2, liq_siglip_zs, width, color=colors["SigLIP 2"], alpha=0.55, label="SigLIP 2 — zero-shot"),
    ax.bar(x[0] + width/2, liq_clip_zs,   width, color=colors["CLIP"],     alpha=0.55, label="CLIP — zero-shot"),
    ax.bar(x[1] - width/2, siglip_ap,     width, color=colors["SigLIP 2"], alpha=1.0,  label="SigLIP 2 — linear probe"),
    ax.bar(x[1] + width/2, clip_ap,       width, color=colors["CLIP"],     alpha=1.0,  label="CLIP — linear probe"),
]
for bar_group in bars:
    for b in bar_group:
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.005,
                f"{b.get_height():.3f}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(["Zero-shot\n(ChangeIt-Frames liquid subset)", "Linear probe\n(AI2-THOR sim crops)"])
ax.set_ylabel("Average Precision (AP)")
ax.set_ylim(0.75, 1.01)
ax.set_title("Zero-shot vs Linear Probe — full/empty state classification")
ax.legend(fontsize=8, loc="lower right")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(RESULTS_PNG / "results_bar.png", dpi=150)
plt.show()

### 5.2  t-SNE of Frozen Embeddings (full vs empty)

In [ ]:
print("Computing t-SNE (~30 s)...")
siglip_2d = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000).fit_transform(siglip_emb)
clip_2d = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000).fit_transform(clip_emb)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
label_colors = {0: "#E66", 1: "#66B"}
label_names  = {0: "empty", 1: "full"}

for ax, emb_2d, title, ap_val in [
    (axes[0], siglip_2d, f"SigLIP 2  (probe AP={siglip_ap:.3f})", siglip_ap),
    (axes[1], clip_2d,   f"CLIP ViT-L/14  (probe AP={clip_ap:.3f})", clip_ap),
]:
    for lbl in [0, 1]:
        mask = sim_labels == lbl
        ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1],
                   c=label_colors[lbl], label=label_names[lbl], alpha=0.6, s=18, linewidths=0)
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle("t-SNE of frozen vision encoder features — full vs empty (AI2-THOR sim crops)", fontsize=11)
fig.tight_layout()
fig.savefig(RESULTS_PNG / "results_tsne.png", dpi=150)
plt.show()

### 5.3  Sample Crops: Correct vs Incorrect (SigLIP 2 probe)

In [ ]:
_, te_indices, _, _ = train_test_split(
    np.arange(len(sim_labels)), sim_labels, test_size=0.2, random_state=42, stratify=sim_labels)

te_images    = [sim_images[i]    for i in te_indices]
te_labels    = sim_labels[te_indices]
te_obj_types = [sim_obj_types[i] for i in te_indices]

correct_idx   = np.where(siglip_preds == siglip_yte)[0]
incorrect_idx = np.where(siglip_preds != siglip_yte)[0]
n_show = min(5, len(correct_idx), len(incorrect_idx))

fig, axes = plt.subplots(2, n_show, figsize=(n_show * 2.4, 5.5))
for col, idx in enumerate(correct_idx[:n_show]):
    ax = axes[0, col]
    ax.imshow(te_images[idx])
    ax.set_title(f"GT:{label_names[te_labels[idx]]}\nPred:{label_names[siglip_preds[col]]}\n{te_obj_types[idx]}",
                 fontsize=7, color="green")
    ax.axis("off")
for col, idx in enumerate(incorrect_idx[:n_show]):
    ax = axes[1, col]
    ax.imshow(te_images[idx])
    ax.set_title(f"GT:{label_names[te_labels[idx]]}\nPred:{label_names[siglip_preds[len(correct_idx[:n_show])+col]]}\n{te_obj_types[idx]}",
                 fontsize=7, color="red")
    ax.axis("off")

axes[0, 0].set_ylabel("Correct",   fontsize=9, color="green", labelpad=6)
axes[1, 0].set_ylabel("Incorrect", fontsize=9, color="red",   labelpad=6)
fig.suptitle("SigLIP 2 linear probe predictions on AI2-THOR test crops", fontsize=11)
fig.tight_layout()
fig.savefig(RESULTS_PNG / "results_crops.png", dpi=150)
plt.show()

## 6  Results Summary

### 6.1  Full/empty: progression across methods

In [ ]:
full_empty_adapter_ap = adapter_df[adapter_df.pair == "full_empty"]["adapter_ap"].values[0]

summary_fe = pd.DataFrame([
    {"method": "Zero-shot (ChangeIt liquid)",       "SigLIP 2 AP": liq_siglip_zs, "CLIP AP": liq_clip_zs},
    {"method": "Linear probe — sim in-domain",      "SigLIP 2 AP": siglip_ap,      "CLIP AP": clip_ap},
    {"method": "Linear probe — sim→real (ChangeIt)","SigLIP 2 AP": s_xd_ap,        "CLIP AP": c_xd_ap},
    {"method": "MLP adapter — sim in-domain",       "SigLIP 2 AP": full_empty_adapter_ap, "CLIP AP": float("nan")},
])
display(summary_fe.style.format("{:.3f}", subset=["SigLIP 2 AP","CLIP AP"])
        .highlight_max(subset=["SigLIP 2 AP","CLIP AP"], axis=0, color="#d4f1c0"))

### 6.2  Adapter: per state-pair breakdown

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
pairs = adapter_df["pair"]
aps   = adapter_df["adapter_ap"]
bar_colors = ["#4C72B0" if ap >= 0.9 else "#DD8452" if ap >= 0.7 else "#CC4444" for ap in aps]
bars = ax.bar(pairs, aps, color=bar_colors)
for b in bars:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01,
            f"{b.get_height():.3f}", ha="center", va="bottom", fontsize=9)
ax.axhline(liq_siglip_zs, color="grey", linestyle="--", linewidth=1, label=f"Zero-shot baseline ({liq_siglip_zs:.3f})")
ax.set_ylabel("Average Precision (AP)")
ax.set_ylim(0, 1.12)
ax.set_title("MLP Adapter — per state-pair AP (SigLIP 2, sim test set)")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=15)
fig.tight_layout()
plt.show()

print(f"\nMean adapter AP: {adapter_df.adapter_ap.mean():.3f}")
print(f"Zero-shot baseline (SigLIP 2 liquid): {liq_siglip_zs:.3f}")

### 6.3  Key findings

**Zero-shot:** Both SigLIP 2 (mean AP 0.935) and CLIP (0.932) are strong on ChangeIt-Frames overall, but both struggle on the liquid-fill subset, which is the hardest state pair (interior not visible from outside).

**Linear probe:** Supervised logistic regression on frozen features improves AP by ~7–8% for SigLIP 2 (0.881 → 0.954) and ~2.4% for CLIP (0.844 → 0.868) in-domain. This confirms that state signal is linearly separable in the frozen feature space, supervision then extracts it.

**Sim-to-real transfer:** AP holds up well (probe trained on sim generalises to real), but accuracy drops sharply for SigLIP 2 (0.90 → 0.46), indicating miscalibration. CLIP's accuracy gap is smaller (0.783 → 0.64), suggesting its features generalise better despite lower in-domain accuracy.

**Adapter:** Easy pairs (dirty/clean, broken/intact: AP 1.0) are trivially solved. Hard pairs (on/off: 0.722, cooked/raw: 0.510) require more and/or better sim data. The adapter justifies the next step: targeted data generation for hard pairs + real-image fine-tuning.